In [8]:
!pip install -q elasticsearch

In [9]:
import json
import os
import pandas as pd
from elasticsearch import Elasticsearch
import numpy as np

In [10]:
import sys
sys.path.append('/kaggle/input/py-lib/AIC')

from elastic_search_processor import ElasticSearchProcessor

In [ ]:
es_processor = ElasticSearchProcessor(
    "https://8641545ce73f41b3a05dbc80de48d72e.asia-northeast1.gcp.cloud.es.io:443", 
    "your api key", 
    "huyrc"
)

success, message = es_processor.create_index_with_mapping()
print(message)

Index 'huyrc' đã tồn tại


In [12]:
es_processor.client.ping()

True

In [13]:
indexMapping = {
    "properties": {
        "video_id": {
            "type": "keyword"
        },
        "pts_time": {
            "type": "float"
        },
        "frame_idx": {
            "type": "float"
        },
        "OCRText": {
            "type": "text"
        }
    }
}
response = es_processor.client.indices.delete(index='ocr_features_first')
response = es_processor.client.indices.delete(index='ocr_features_second')
response = es_processor.client.indices.delete(index='ocr_features_3')

es_processor.client.indices.create(index="ocr_features_first", mappings=indexMapping)
es_processor.client.indices.create(index="ocr_features_second", mappings=indexMapping)
es_processor.client.indices.create(index="ocr_features_3", mappings=indexMapping)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'ocr_features_3'})

In [14]:
# %%
from elasticsearch import helpers
#folder_path = '/kaggle/input/ocr-final/OCR'
csv_mapping_path = '/kaggle/input/new-index-final/csv_mapping'
json_file_path = '/kaggle/input/ocr-all/OCR_all.json'
# %%
batch_size = 1000  # Number of documents to send in each bulk request
documents_first = []
#documents_second = []
#documents_third = []
#i = 0
temp = None
with open(json_file_path, 'r') as json_file:
    ocr_data = json.load(json_file)
for key, ocr_text_list in ocr_data.items():
    # Extract the video ID and frame number from the key (e.g., "L01_V001/0001.jpg")
    video_id = key.split('/')[-2]  # e.g., L01_V001
    if video_id != temp:
        print("Start processing video: ", {video_id})
        temp = video_id
    #print(video_path)
    #video_id = video_path.split('/')[-1]  # Extract video_id (e.g., L01_V001)
    frame_number = int(key.split('/')[-1].replace('.jpg', ''))  # Extract frame number (e.g., 0001)

    # Load the corresponding CSV file
    csv_file_path = os.path.join(csv_mapping_path, f'{video_id}.csv')
    csv_data = pd.read_csv(csv_file_path)

    # Find the matching row for the current frame number
    matching_row = csv_data[csv_data['n'] == frame_number]
    if not matching_row.empty:
        pts_time = matching_row['pts_time'].values[0]
        frame_idx = matching_row['frame_idx'].values[0]
        
        # Index each OCR text in the Elasticsearch document
        for ocr_text in ocr_text_list:
            doc = {
                "_index": "ocr_features_first",
                "_source": {
                    "video_id": video_id,
                    "pts_time": pts_time,
                    "frame_idx": frame_idx,
                    "OCRText": ocr_text
                }
            }
            documents_first.append(doc)

            # Send bulk requests when the batch size is reached
            if len(documents_first) >= batch_size:
                helpers.bulk(es_processor.client, documents_first)
                documents_first = []  # Reset the list after bulk indexing
    #print(f"Finished processing video: {video_id}")

# Final bulk indexing for remaining documents
if documents_first:
    helpers.bulk(es_processor.client, documents_first)

print("All videos processed!")
'''
for file_name in os.listdir(folder_path): 
    if file_name.endswith('.json') and file_name.startswith('L02') or file_name.startswith('L01'):
        file_path = os.path.join(folder_path, file_name)
        video_id = file_name.replace('.json', '')
        
        with open(file_path, 'r') as json_file:
            ocr_data = json.load(json_file)
        csv_file_path = os.path.join(csv_mapping_path, f'{video_id}.csv')
        csv_data = pd.read_csv(csv_file_path)
        
        for key, ocr_text_list in ocr_data.items():
            frame_number = int(key.split('/')[-1].replace('.jpg', ''))
            matching_row = csv_data[csv_data['n'] == frame_number]
            if not matching_row.empty:
                pts_time = matching_row['pts_time'].values[0]
                frame_idx = matching_row['frame_idx'].values[0]
                for ocr_text in ocr_text_list:
                    doc = {
                        "_index": "ocr_features_first",# if i < 10000 else "ocr_features_second" if i < 20000 else "ocr_features_3",
                        "_source": {
                            "video_id": video_id,
                            "pts_time": pts_time,
                            "frame_idx": frame_idx,
                            "OCRText": ocr_text
                        }
                    }

                    #if i < 10000:
                    documents_first.append(doc)
                    #elif i >= 10000 and i < 20000:
                    #    documents_second.append(doc)
                    #else:
                    #    documents_third.append(doc)

                    #i += 1

                    # Send bulk requests when the batch size is reached
                    if len(documents_first) >= batch_size:
                        helpers.bulk(es_processor.client, documents_first)
                        documents_first = []  # Reset the list after bulk indexing

                    #if len(documents_second) >= batch_size:
                    #    helpers.bulk(es_processor.client, documents_second)
                    #    documents_second = []  # Reset the list after bulk indexing

                    #if len(documents_third) >= batch_size:
                    #    helpers.bulk(es_processor.client, documents_third)
                    #    documents_third = []  # Reset the list after bulk indexing

        print("Finished indexing one video!")

# Final bulk indexing for remaining documents
if documents_first:
    helpers.bulk(es_processor.client, documents_first)

#if documents_second:
#    helpers.bulk(es_processor.client, documents_second)

#if documents_third:
#    helpers.bulk(es_processor.client, documents_third)

print("All videos processed!")'''

Start processing video:  {'L01_V001'}
Start processing video:  {'L01_V002'}
Start processing video:  {'L01_V003'}
Start processing video:  {'L01_V004'}
Start processing video:  {'L01_V005'}
Start processing video:  {'L01_V006'}
Start processing video:  {'L01_V007'}
Start processing video:  {'L01_V008'}
Start processing video:  {'L01_V009'}
Start processing video:  {'L01_V010'}
Start processing video:  {'L01_V011'}
Start processing video:  {'L01_V012'}
Start processing video:  {'L01_V013'}
Start processing video:  {'L01_V014'}
Start processing video:  {'L01_V015'}
Start processing video:  {'L01_V016'}
Start processing video:  {'L01_V017'}
Start processing video:  {'L01_V018'}
Start processing video:  {'L01_V019'}
Start processing video:  {'L01_V020'}
Start processing video:  {'L01_V021'}
Start processing video:  {'L01_V022'}
Start processing video:  {'L01_V023'}
Start processing video:  {'L01_V024'}
Start processing video:  {'L01_V025'}
Start processing video:  {'L01_V026'}
Start proces

'\nfor file_name in os.listdir(folder_path): \n    if file_name.endswith(\'.json\') and file_name.startswith(\'L02\') or file_name.startswith(\'L01\'):\n        file_path = os.path.join(folder_path, file_name)\n        video_id = file_name.replace(\'.json\', \'\')\n        \n        with open(file_path, \'r\') as json_file:\n            ocr_data = json.load(json_file)\n        csv_file_path = os.path.join(csv_mapping_path, f\'{video_id}.csv\')\n        csv_data = pd.read_csv(csv_file_path)\n        \n        for key, ocr_text_list in ocr_data.items():\n            frame_number = int(key.split(\'/\')[-1].replace(\'.jpg\', \'\'))\n            matching_row = csv_data[csv_data[\'n\'] == frame_number]\n            if not matching_row.empty:\n                pts_time = matching_row[\'pts_time\'].values[0]\n                frame_idx = matching_row[\'frame_idx\'].values[0]\n                for ocr_text in ocr_text_list:\n                    doc = {\n                        "_index": "ocr_featu

In [16]:
# Encode the input keyword
query = "NGAP NANG TAI MIEN TRUNG"
doc_count_1 = es_processor.client.count(index='ocr_features_first')['count']
doc_count_2 = es_processor.client.count(index='ocr_features_second')['count']
doc_count_3 = es_processor.client.count(index='ocr_features_3')['count']

# KNN search query
query_text_1 = {
    "size": 10,  # Return top 10 results
    "query": {
        "match": {
            "OCRText": query
        }
    }
}

query_text_2 = {
    "size": 10,  # Return top 10 results
    "query": {
        "match": {
            "OCRText": query
        }
    }
}

query_text_3 = {
    "size": 10,  # Return top 10 results
    "query": {
        "match": {
            "OCRText": query
        }
    }
}
# Perform the search
print("First part:")
res = es_processor.client.search(index="ocr_features_first", body=query_text_1, _source=["video_id", "pts_time", "frame_idx", "OCRText"])
for hit in res["hits"]["hits"]:
    print(f"Video ID: {hit['_source']['video_id']}, PTS_Time: {hit['_source']['pts_time']}, Frame_idx: {hit['_source']['frame_idx']}, OCR_text: {hit['_source']['OCRText']}")
print("Second part:")
res = es_processor.client.search(index="ocr_features_second", body=query_text_2, _source=["video_id", "pts_time", "frame_idx"])
for hit in res["hits"]["hits"]:
    print(f"Video ID: {hit['_source']['video_id']}, PTS_Time: {hit['_source']['pts_time']}, Frame_idx: {hit['_source']['frame_idx']}")
print("Third_part:")
res = es_processor.client.search(index="ocr_features_3", body=query_text_3, _source=["video_id", "pts_time", "frame_idx"])
for hit in res["hits"]["hits"]:
    print(f"Video ID: {hit['_source']['video_id']}, PTS_Time: {hit['_source']['pts_time']}, Frame_idx: {hit['_source']['frame_idx']}")

First part:
Video ID: L02_V018, PTS_Time: 16.72, Frame_idx: 418, OCR_text: NGAP NANG TAI MIEN TRUNG
Video ID: L02_V018, PTS_Time: 16.76, Frame_idx: 419, OCR_text: NGAP NANG TAI MIEN TRUNG
Video ID: L02_V018, PTS_Time: 20.12, Frame_idx: 503, OCR_text: NGP NANG TAI MIEN TRUNG
Video ID: L02_V018, PTS_Time: 15.88, Frame_idx: 397, OCR_text: MUA LON GAY LU LYT VA NGAP NANG TAI MIEN TRUNG
Video ID: L02_V018, PTS_Time: 18.4, Frame_idx: 460, OCR_text: MUA LON GAY LU LYT VA NGAP NANG TAI MIEN TRUNG
Video ID: L02_V018, PTS_Time: 20.16, Frame_idx: 504, OCR_text: MUA LON GAY LU LYT VA NGAP NANG TAI MIEN TRUNG
Video ID: L20_V007, PTS_Time: 649.5, Frame_idx: 19485, OCR_text: Mien Trung nang nong dien
Video ID: L20_V007, PTS_Time: 649.5333333333333, Frame_idx: 19486, OCR_text: Mien Trung nang nong dien
Video ID: L20_V007, PTS_Time: 968.6666666666666, Frame_idx: 29060, OCR_text: Mien Trung nang nong dien
Video ID: L02_V017, PTS_Time: 941.24, Frame_idx: 23531, OCR_text: nha bi ngap lut tgi mien Trung
Se